In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression

# 2. Load dataset
df = pd.read_csv("loan_data_set.csv")

# 3. Check missing values
print("Missing values:\n", df.isnull().sum())

# =====================================================
# NUMERICAL IMPUTATION METHODS
# =====================================================

# 4. Mean Imputation
df_mean = df.copy()
df_mean['LoanAmount'] = df_mean['LoanAmount'].fillna(df_mean['LoanAmount'].mean())

# 5. Median Imputation
df_median = df.copy()
df_median['LoanAmount'] = df_median['LoanAmount'].fillna(df_median['LoanAmount'].median())

# 6. Mode Imputation
df_mode = df.copy()
df_mode['LoanAmount'] = df_mode['LoanAmount'].fillna(df_mode['LoanAmount'].mode()[0])

# =====================================================
# CATEGORICAL IMPUTATION
# =====================================================

# 7. Fill categorical missing values with most frequent value
df_cat = df.copy()
df_cat['Gender'] = df_cat['Gender'].fillna(df_cat['Gender'].mode()[0])
df_cat['Self_Employed'] = df_cat['Self_Employed'].fillna(df_cat['Self_Employed'].mode()[0])
df_cat['Credit_History'] = df_cat['Credit_History'].fillna(df_cat['Credit_History'].mode()[0])

# =====================================================
# RANDOM SAMPLE IMPUTATION
# =====================================================

df_random = df.copy()

missing_count = df_random['LoanAmount'].isnull().sum()

sample_vals = df_random['LoanAmount'].dropna().sample(missing_count, random_state=0)

df_random.loc[df_random['LoanAmount'].isnull(), 'LoanAmount'] = sample_vals.values

# =====================================================
# ENCODING CATEGORICAL VARIABLES
# =====================================================

df_enc = df.copy()

cat_cols = df_enc.select_dtypes(include='object').columns

encoder = OrdinalEncoder()
df_enc[cat_cols] = encoder.fit_transform(df_enc[cat_cols])

# =====================================================
# REGRESSION IMPUTATION
# =====================================================

df_reg = df.copy()

# Split data into known & missing
train = df_reg[df_reg['LoanAmount'].notnull()]
test = df_reg[df_reg['LoanAmount'].isnull()]

# Features & target
X_train = train[['CoapplicantIncome']]
y_train = train['LoanAmount']

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict missing values
predicted_values = model.predict(test[['CoapplicantIncome']])

# Fill missing values
df_reg.loc[df_reg['LoanAmount'].isnull(), 'LoanAmount'] = predicted_values

# =====================================================
print("All imputations completed ✔️")

Missing values:
 Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64
All imputations completed ✔️


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression

# Load dataset
df = pd.read_csv("loan_data_set.csv")

print("Missing values:\n", df.isnull().sum())

# NUMERICAL IMPUTATION (all in one dictionary)
df_impute = df.copy()

df_impute['LoanAmount_mean']   = df['LoanAmount'].fillna(df['LoanAmount'].mean())
df_impute['LoanAmount_median'] = df['LoanAmount'].fillna(df['LoanAmount'].median())
df_impute['LoanAmount_mode']   = df['LoanAmount'].fillna(df['LoanAmount'].mode()[0])

# CATEGORICAL IMPUTATION (loop instead of repeating)
cat_cols = ['Gender', 'Self_Employed', 'Credit_History']

for col in cat_cols:
    df_impute[col] = df_impute[col].fillna(df_impute[col].mode()[0])

# RANDOM SAMPLE IMPUTATION
missing = df['LoanAmount'].isnull()

df_impute.loc[missing, 'LoanAmount_random'] = (
    df['LoanAmount'].dropna().sample(missing.sum(), random_state=0).values
)

# ENCODING
encoder = OrdinalEncoder()
obj_cols = df_impute.select_dtypes(include='object').columns
df_impute[obj_cols] = encoder.fit_transform(df_impute[obj_cols])

# REGRESSION IMPUTATION
train = df[df['LoanAmount'].notnull()]
test  = df[df['LoanAmount'].isnull()]

model = LinearRegression()
model.fit(train[['CoapplicantIncome']], train['LoanAmount'])

df_impute.loc[df['LoanAmount'].isnull(), 'LoanAmount_reg'] = (
    model.predict(test[['CoapplicantIncome']])
)

print("All imputations completed ✔️")

Missing values:
 Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64
All imputations completed ✔️
